In [63]:
# %pip install git+https://github.com/alemartinello/dstapi
import requests
import pandas as pd
from IPython.display import display
from io import StringIO
from dstapi import DstApi
from functools import reduce

# `Expenditure shares`

Types of investment expenditures as a fraction of GDP
- **Source:** IO
- **Structure:** include 60% of spending on:
    - Administrative and support services”
    - Miscellaneous professional, scientific, and technical services
    - Management of companies and enterprises
- **Equipment:** 
    - Accounting, consulting, design, and computer services.
- **Intellectual property:** 
    - Service output of establishments that manage other units within a company.
- **Organizational services:** 
    - Expenditures related to workforce human capital, distribution systems, logistics, product design, and customer and brand capital.

### 1. Investment expenditure

- **DST table for investment by asset type: NAHI:** Gross capital formation by assets, price unit and time
- **DST table for GDP:** Use NAN1: Demand and supply by transaction and price unit.

In [68]:
# 1. get using DST API
NAHI = DstApi('NAHI')
# NAHI.tablesummary(language='en')
# NAHI.variable_levels('AKTIV',language='en')

# 2. set up fetch
params_I = {
    'table': 'NAHI',
    'format': 'BULK',
    'lang': 'en',
    'variables': [
        {'code': 'PRISENHED', 'values': ['V']},   # 2020-prices, chained values
        {'code': 'Tid', 'values': ['*']},
        {'code': 'AKTIV', 'values': ['*']}
        ]
}
params_Y = {
    'table': 'NAN1',
    'format': 'BULK',
    'lang': 'en',
    'variables': [
        {'code': 'PRISENHED', 'values': ['V_M']},   # 2020-prices, chained values
        {'code': 'Tid', 'values': ['*']},
        {'code': 'TRANSAKT', 'values': ['B1GQK']}
        ]
}

# 3. get data
df_I = NAHI.get_data(params=params_I)
df_Y = NAHI.get_data(params=params_Y)
df_I['INDHOLD'] = pd.to_numeric(df_I['INDHOLD'], errors='coerce')
df_Y['INDHOLD'] = pd.to_numeric(df_Y['INDHOLD'], errors='coerce')

# 4. helper to sum a set of AKTIV each year
def sum_aktiv(df, aktiv_list, colname):
    return (df.loc[df['AKTIV'].isin(aktiv_list), ['TID','INDHOLD']]
              .groupby('TID', as_index=False)['INDHOLD'].sum()
              .rename(columns={'INDHOLD': colname}))

struc_list = ['Dwellings','Buildings other than dwellings','Other structures and land improvements']
equip_list = ['ICT equipment, other machinery and equipment and weapon systems','Transport equipment']
IPP_list   = ['Intellectual property products']

# 5. investment aggregates by year
I_struc = sum_aktiv(df_I, struc_list, 'I_struc')
I_equip = sum_aktiv(df_I, equip_list, 'I_equip')
I_ipp   = sum_aktiv(df_I, IPP_list,   'I_ipp')

GDP = (df_Y.loc[df_Y['TRANSAKT'].eq('B1GQK'), ['TID','INDHOLD']]
         .groupby('TID', as_index=False)['INDHOLD'].sum()
         .rename(columns={'INDHOLD': 'GDP'}))

# 7. merge + compute shares
df = reduce(lambda left, right: pd.merge(left, right, on='TID', how='inner'),
            [GDP, I_struc, I_equip, I_ipp])

# for c in ['I_struc','I_equip','I_ipp']:
#     df[c + '_share'] = df[c] / df['GDP']          # level share
#     df[c + '_pct']   = 100 * df[c + '_share']     # percent

# df = df.sort_values('TID').reset_index(drop=True)
GDP

,TID,GDP
